# Ingesta de Datos via API - Registro Mercantil RUES

## Objetivo

Este notebook ingesta datos desde una **API REST (Socrata)** de datos.gov.co hacia el catálogo de la materia **Gestión de Datos Masivos**, siguiendo la misma estructura y convenciones definidas en `Configuracion_inicial.ipynb`.

## Dataset

* **Nombre**: Personas Naturales, Personas Jurídicas y Entidades Sin Ánimo de Lucro (RUES)
* **Descripción**: Extracción de los datos básicos del Registro Mercantil que administran las Cámaras de Comercio a nivel nacional, sincronizados en el Registro Único Empresarial y Social (RUES).
* **Categoría**: Comercio, Industria y Turismo
* **Fuente**: [datos.gov.co - recurso c82u-588k](https://www.datos.gov.co/resource/c82u-588k.json)
* **Filtro**: solo registros con `fecha_actualizacion` en los últimos 2 años
* **Columnas**: 36 (todas tipo texto) — ver detalle al final del notebook

## Ubicación destino

**Catálogo**: `Datos_Empresas`  
**Esquema**: `bronze` (capa de datos crudos, arquitectura Medallion)  
**Tabla**: `gdm_estructurados_registro_mercantil_api`  
**Clasificación**: Datos **estructurados** (esquema fijo y tabular), aunque la fuente de ingesta sea una API JSON

---

In [ ]:
%sql
-- Verificar catálogos y esquemas disponibles antes de crear la tabla
SHOW CATALOGS;

In [ ]:
%sql
-- Crear catálogo para la materia de Gestión de Datos Masivos (idempotente)
-- Dedicado al análisis de información empresarial de Colombia (RUES)

CREATE CATALOG IF NOT EXISTS Datos_Empresas
COMMENT 'Catálogo de información empresarial de Colombia para la materia de Gestión de Datos Masivos. Reúne datos del Registro Único Empresarial y Social (RUES) — personas naturales, personas jurídicas y entidades sin ánimo de lucro registradas ante las Cámaras de Comercio — obtenidos mediante la API abierta de datos.gov.co (recurso c82u-588k). Sirve como caso de estudio para la ingesta de datos estructurados desde una fuente REST/JSON hacia una arquitectura Medallion (Bronze/Silver/Gold).';

In [ ]:
%sql
-- Confirmar que el esquema default existe en el catálogo workspace
SHOW SCHEMAS IN workspace;

---

## Paso 1: Consultar el total de registros disponibles en la API

Socrata permite usar SoQL (`$select=count(*)`) para conocer el total de filas antes de descargar. Además, filtramos con `$where` para traer solo los registros actualizados en los **últimos 2 años** (campo `fecha_actualizacion`), lo que reduce el volumen a descargar.

In [ ]:
%python
import requests
import pandas as pd
from datetime import datetime, timedelta

BASE_URL = "https://www.datos.gov.co/resource/c82u-588k.json"
LIMIT = 50000  # tamaño de página soportado por Socrata

# fecha_actualizacion llega como texto 'YYYY/MM/DD HH:MM:SS...', comparable como string
FECHA_CORTE = (datetime.now() - timedelta(days=365 * 2)).strftime("%Y/%m/%d")
WHERE_CLAUSE = f"fecha_actualizacion >= '{FECHA_CORTE}'"


def contar_registros(where=None):
    params = {"$select": "count(*)"}
    if where:
        params["$where"] = where
    resp = requests.get(BASE_URL, params=params)
    resp.raise_for_status()
    return int(resp.json()[0]["count"])


total_registros = contar_registros(WHERE_CLAUSE)
print(f"Fecha de corte (últimos 2 años): {FECHA_CORTE}")
print(f"Total de registros actualizados desde esa fecha: {total_registros}")

---

## Paso 2: Descargar los datos paginando

> Nota: Ya filtrado a los últimos 2 años (`WHERE_CLAUSE`), pero aun así puede ser un volumen grande. Para pruebas rápidas, ajusta `MAX_REGISTROS` a un número pequeño (por ejemplo `100000`). Para la ingesta completa del rango filtrado, usa `total_registros`.

In [ ]:
%python
# Cambia este valor por total_registros para descargar todo el rango filtrado (últimos 2 años)
MAX_REGISTROS = total_registros


def descargar_datos(max_registros, where=None):
    registros = []
    offset = 0

    while offset < max_registros:
        pagina_limit = min(LIMIT, max_registros - offset)
        params = {"$limit": pagina_limit, "$offset": offset}
        if where:
            params["$where"] = where
        resp = requests.get(BASE_URL, params=params)
        resp.raise_for_status()
        pagina = resp.json()

        if not pagina:
            break

        registros.extend(pagina)
        offset += LIMIT
        print(f"Descargados {len(registros)} de {max_registros} registros...")

    return registros


registros = descargar_datos(MAX_REGISTROS, WHERE_CLAUSE)
df_pandas = pd.DataFrame(registros)
print(f"\nDataset descargado: {df_pandas.shape[0]} filas x {df_pandas.shape[1]} columnas")
df_pandas.head()

---

## Paso 3: Convertir a Spark DataFrame y crear vista temporal

In [ ]:
%python
# Todas las columnas de la API llegan como texto (string), lo cual es
# consistente con el tipo de dato declarado por Socrata para este dataset
df_spark = spark.createDataFrame(df_pandas.astype(str))
df_spark.createOrReplaceTempView("temp_registro_mercantil_api")

display(df_spark.limit(10))

---

## Paso 4: Crear la tabla Delta en el catálogo

Siguiendo la convención de nomenclatura `gdm_<tipo>_<nombre>` definida en `Configuracion_inicial.ipynb`.

In [ ]:
%sql
-- Nota: temp_registro_mercantil_api se crea en la celda de Python anterior (Paso 3)
CREATE OR REPLACE TABLE Datos_Empresas.bronze.gdm_estructurados_registro_mercantil_api
USING DELTA
COMMENT 'Personas Naturales, Personas Jurídicas y Entidades Sin Ánimo de Lucro (RUES). Datos básicos del Registro Mercantil administrado por las Cámaras de Comercio a nivel nacional. Ingestado vía API Socrata de datos.gov.co (recurso c82u-588k).'
AS
SELECT * FROM temp_registro_mercantil_api;

-- Verificar que la tabla se creó exitosamente
DESCRIBE EXTENDED Datos_Empresas.bronze.gdm_estructurados_registro_mercantil_api;

In [ ]:
%sql
-- Verificación rápida de la ingesta
SELECT COUNT(*) AS total_filas FROM Datos_Empresas.bronze.gdm_estructurados_registro_mercantil_api;

SELECT * FROM Datos_Empresas.bronze.gdm_estructurados_registro_mercantil_api LIMIT 10;

---

## Próximos pasos

1. Si `MAX_REGISTROS` se usó para una prueba parcial, vuelve a ejecutar el Paso 2 con `total_registros` para la ingesta completa.
2. Para descargas grandes (varios millones de filas), considera aumentar `LIMIT`, o descargar por lotes y hacer `INSERT INTO` incremental en vez de `CREATE OR REPLACE TABLE`.
3. Explora una capa **Silver**: limpieza de tipos (fechas en `fecha_matricula`, `fecha_renovacion`, etc. llegan como texto y deberían convertirse a `DATE`), normalización de texto, deduplicación por `matricula` + `codigo_camara`.

### Columnas del dataset (36, todas `text` en origen)

`codigo_camara`, `camara_comercio`, `matricula`, `inscripcion_proponente`, `razon_social`, `primer_apellido`, `segundo_apellido`, `primer_nombre`, `segundo_nombre`, `sigla`, `codigo_clase_identificacion`, `clase_identificacion`, `numero_identificacion`, `nit`, `digito_verificacion`, `cod_ciiu_act_econ_pri`, `cod_ciiu_act_econ_sec`, `ciiu3`, `ciiu4`, `fecha_matricula`, `fecha_renovacion`, `ultimo_ano_renovado`, `fecha_vigencia`, `fecha_cancelacion`, `codigo_tipo_sociedad`, `tipo_sociedad`, `codigo_organizacion_juridica`, `organizacion_juridica`, `codigo_categoria_matricula`, `categoria_matricula`, `codigo_estado_matricula`, `estado_matricula`, `clase_identificacion_RL`, `num_identificacion_representante_legal`, `representante_legal`, `fecha_actualizacion`